# 🦷 DentiAI - High-Precision Dental YOLO11m Training on Kaggle
This notebook trains a high-resolution (**1280px**), medical-grade **YOLO11m** detector on the **DENTEX** panoramic radiograph dataset using a free 16 GB Kaggle GPU (Tesla T4 / P100).

### Target Objectives:
- **Resolution**: $1280\times 1280$ with `rect=True` (preserves tooth aspect ratio without blurring fine caries)
- **Model**: `YOLO11m` (20.1M parameters - much deeper representations than YOLO11s)
- **Precision Focus**: Medical augmentations (grayscale invariance, restricted mosaic, AdamW)
- **Output**: `best.pt` ready for 1-click download to your local DentiAI project.

In [ ]:
# 1. Verify GPU Hardware and VRAM
!nvidia-smi

In [ ]:
# 2. Install Ultralytics and dependencies
!pip install -q --upgrade ultralytics opencv-python-headless
import torch
from ultralytics import YOLO
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} (VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB)")

In [ ]:
# 3. Locate or Prepare the DENTEX Dataset in YOLO format
import os
import glob
import json
import random
import shutil

dataset_dir = "/kaggle/working/dentex_yolo"
os.makedirs(dataset_dir, exist_ok=True)

# Search for DENTEX dataset in /kaggle/input/
search_json = glob.glob("/kaggle/input/**/train_quadrant_enumeration_disease.json", recursive=True)
search_zip = glob.glob("/kaggle/input/**/dentex*.zip", recursive=True)

if search_json:
    json_path = search_json[0]
    images_dir = os.path.join(os.path.dirname(json_path), "xrays")
    print(f"Found DENTEX annotations at: {json_path}")
    print(f"Found DENTEX images at: {images_dir}")
    
    # Convert COCO-style DENTEX format to YOLO
    random.seed(42)
    with open(json_path, "r") as f:
        data = json.load(f)
        
    images = data["images"]
    annotations = data["annotations"]
    
    img_to_anns = {}
    for ann in annotations:
        img_to_anns.setdefault(ann["image_id"], []).append(ann)
        
    id_to_img = {img["id"]: img for img in images}
    all_img_ids = list(id_to_img.keys())
    random.shuffle(all_img_ids)
    
    val_count = int(len(all_img_ids) * 0.2)
    val_ids = set(all_img_ids[:val_count])
    
    for split in ["train", "val"]:
        os.makedirs(os.path.join(dataset_dir, "images", split), exist_ok=True)
        os.makedirs(os.path.join(dataset_dir, "labels", split), exist_ok=True)
        
    print("Formatting dataset for YOLO...")
    for img_id, img_info in id_to_img.items():
        split = "val" if img_id in val_ids else "train"
        file_name = img_info["file_name"]
        src_img_path = os.path.join(images_dir, file_name)
        dst_img_path = os.path.join(dataset_dir, "images", split, file_name)
        
        if not os.path.exists(dst_img_path) and os.path.exists(src_img_path):
            os.symlink(src_img_path, dst_img_path)
            
        base_name = os.path.splitext(file_name)[0]
        label_file = os.path.join(dataset_dir, "labels", split, f"{base_name}.txt")
        
        img_w = float(img_info["width"])
        img_h = float(img_info["height"])
        
        lines = []
        for ann in img_to_anns.get(img_id, []):
            cat_id = ann["category_id_3"]  # 0: Impacted, 1: Caries, 2: Periapical Lesion, 3: Deep Caries
            x, y, w, h = ann["bbox"]
            x_c = max(0.0, min(1.0, (x + w / 2.0) / img_w))
            y_c = max(0.0, min(1.0, (y + h / 2.0) / img_h))
            n_w = max(0.0, min(1.0, w / img_w))
            n_h = max(0.0, min(1.0, h / img_h))
            lines.append(f"{cat_id} {x_c:.6f} {y_c:.6f} {n_w:.6f} {n_h:.6f}")
            
        with open(label_file, "w") as lf:
            lf.write("\n".join(lines))
            
elif search_zip:
    print(f"Unpacking uploaded zip: {search_zip[0]}")
    !unzip -q {search_zip[0]} -d /kaggle/working/
    dataset_dir = glob.glob("/kaggle/working/**/dentex_yolo", recursive=True)[0]
else:
    print("Note: Please attach the DENTEX dataset or upload dentex_yolo.zip in /kaggle/input/")

# Write data.yaml
yaml_path = "/kaggle/working/data.yaml"
yaml_content = f"""path: {dataset_dir}
train: images/train
val: images/val

names:
  0: Impacted
  1: Caries
  2: Periapical Lesion
  3: Deep Caries
"""
with open(yaml_path, "w") as f:
    f.write(yaml_content)
print(f"Dataset successfully configured! YAML written to {yaml_path}")

In [ ]:
# 4. Launch High-Resolution YOLO11m Training (1280px, AdamW)
model = YOLO("yolo11m.pt")  # 20.1M parameters - optimal accuracy/speed balance

results = model.train(
    data="/kaggle/working/data.yaml",
    epochs=70,
    patience=15,
    imgsz=1280,             # Full high resolution on 16GB GPU
    rect=True,              # Panoramic aspect ratio preservation (2:1)
    batch=8,                # Stable gradient descent without OOM
    device=0,
    workers=4,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    fliplr=0.5,             # Jaw horizontal symmetry
    flipud=0.0,
    degrees=4.0,
    hsv_h=0.0,              # Radiograph grayscale invariance
    hsv_s=0.0,
    hsv_v=0.35,             # X-ray contrast/exposure variations
    mosaic=0.3,             # Preserves dental anatomical alignment
    close_mosaic=12,        # Clean morphology in final training epochs
    project="dentiai_kaggle",
    name="yolo11m_dentex_highres1280",
    exist_ok=True,
    plots=True,
    save=True
)
print("Training completed successfully!")

In [ ]:
# 5. Detailed High-Precision Validation & Metrics
metrics = model.val(data="/kaggle/working/data.yaml", device=0, imgsz=1280, conf=0.25)
print("\n--- Overall Validation Metrics ---")
print(f"mAP@50    : {metrics.box.map50:.4f}")
print(f"mAP@50-95 : {metrics.box.map:.4f}")

print("\n--- Per-Class Precision & Recall at default conf=0.25 ---")
for i, name in metrics.names.items():
    p = metrics.box.p[i] if i < len(metrics.box.p) else 0
    r = metrics.box.r[i] if i < len(metrics.box.r) else 0
    ap50 = metrics.box.ap50[i] if i < len(metrics.box.ap50) else 0
    print(f"{name:<18} | Precision: {p*100:5.1f}% | Recall: {r*100:5.1f}% | mAP50: {ap50*100:5.1f}%")

# Copy best weights to root of working directory for easy download
!cp /kaggle/working/dentiai_kaggle/yolo11m_dentex_highres1280/weights/best.pt /kaggle/working/denti_yolo11m_best.pt
print("\nDone! You can now download /kaggle/working/denti_yolo11m_best.pt")